# Tool Evaluation Metrics in Agentic AI

## What Are We Evaluating?

Tools are the **"hands"** of an agent — they interact with the real world (APIs, databases, web).  
If tools work incorrectly, the entire agent pipeline produces wrong results.

```
Agent Brain (LLM)
    ↓ decides which tool to call
    ↓ decides what arguments to pass
Tool Execution  ← THIS is what we evaluate
    ↓ returns result
Agent uses result → STM → next agent/tool
```

## The 6 Core Tool Metrics

| # | Metric | What It Measures | Good Score |
|---|---|---|---|
| 1 | **Tool Selection Accuracy** | Right tool chosen? | > 0.90 |
| 2 | **Tool Argument Accuracy** | Correct params passed? | > 0.95 |
| 3 | **Tool Call Efficiency** | No redundant calls? | > 0.80 |
| 4 | **Tool Hallucination Rate** | Invented tool names/outputs? | < 0.05 |
| 5 | **Tool Success Rate** | Calls completed without error? | > 0.95 |
| 6 | **Tool Output Utilization** | Output actually used in answer? | > 0.85 |

## Why Tool Metrics Matter

```
Bad tool call → wrong data in STM → next agent reads bad data → cascade failure

Example:
  PatientDB returns "Metformin 500mg"
  LLM hallucinates argument → calls PatientDB("wrong_patient_id")
  PatientDB returns empty/wrong → DrugInteraction gets wrong drug
  RiskCalc gets wrong risk → Report is WRONG → Doctor gets wrong recommendation
```

## Step 1 — Install & Setup

In [ ]:
!pip install langchain-openai pandas tabulate

## Step 2 — Tool Registry & Execution Logger

The foundation of all tool metrics is a **tracked tool executor** that logs every call.

In [ ]:
import time
import uuid
from datetime import datetime

# ── Simulated Tools (Medical Agent use case) ─────────────────────────────────
def tool_patient_db(patient_name: str, record_type: str = "all") -> dict:
    """Query patient medical records."""
    db = {
        "John": {
            "medications" : "Metformin 500mg twice daily, Amlodipine 5mg",
            "lab_results" : "HbA1c: 8.1%, eGFR: 78",
            "bp_history"  : "145/92 → 150/95 → 148/93",
            "all"         : "John, 45M, T2DM + Stage1 HTN. Meds: Metformin 500mg, Amlodipine 5mg"
        }
    }
    result = db.get(patient_name, {}).get(record_type)
    if result is None:
        raise ValueError(f"Patient '{patient_name}' or record_type '{record_type}' not found")
    return {"patient": patient_name, "record_type": record_type, "data": result}

def tool_drug_interaction(drug1: str, drug2: str) -> dict:
    """Check drug interaction between two medications."""
    known = {("Metformin", "Amlodipine"): "No major interaction. Monitor potassium."}
    result = known.get((drug1, drug2)) or known.get((drug2, drug1)) or "No known interaction"
    return {"drug1": drug1, "drug2": drug2, "interaction": result}

def tool_web_search(query: str) -> dict:
    """Search the web for medical information."""
    return {"query": query, "result": f"Top result for '{query}': Latest 2026 guidelines recommend..."}

def tool_risk_calculator(age: int, hba1c: float, bp_systolic: int) -> dict:
    """Calculate 10-year cardiovascular risk."""
    risk = round(10 + (age - 40) * 0.5 + (hba1c - 7) * 2 + (bp_systolic - 120) * 0.1, 1)
    return {"10yr_cvd_risk": f"{risk}%", "level": "HIGH" if risk > 20 else "MODERATE"}

# ── Tool Registry — the ONLY valid tools ─────────────────────────────────────
TOOL_REGISTRY = {
    "patient_db"       : tool_patient_db,
    "drug_interaction" : tool_drug_interaction,
    "web_search"       : tool_web_search,
    "risk_calculator"  : tool_risk_calculator,
}

# ── Execution Log — records every tool call ───────────────────────────────────
execution_log = []

def tracked_tool_call(tool_name: str, args: dict, expected_tool: str = None) -> dict:
    """
    Execute a tool and log the call for metric computation.
    expected_tool: what the correct tool SHOULD have been (for labeled evaluation)
    """
    entry = {
        "call_id"       : str(uuid.uuid4())[:8],
        "tool_name"     : tool_name,
        "args"          : args,
        "expected_tool" : expected_tool,
        "timestamp"     : datetime.now().isoformat(),
        "success"       : False,
        "result"        : None,
        "error"         : None,
        "latency_ms"    : 0,
        "is_hallucinated": tool_name not in TOOL_REGISTRY,
        "is_correct_tool": tool_name == expected_tool if expected_tool else None,
        "output_used_in_answer": None,  # filled in later
    }

    # Check: is it a real tool?
    if tool_name not in TOOL_REGISTRY:
        entry["error"] = f"HALLUCINATED TOOL: '{tool_name}' does not exist"
        print(f"  🚨 HALLUCINATION: Tool '{tool_name}' not in registry!")
        execution_log.append(entry)
        return entry

    # Execute the real tool
    start = time.time()
    try:
        result = TOOL_REGISTRY[tool_name](**args)
        entry["success"] = True
        entry["result"]  = result
        print(f"  ✅ Tool '{tool_name}' succeeded in {int((time.time()-start)*1000)}ms")
    except Exception as e:
        entry["error"]   = str(e)
        entry["success"] = False
        print(f"  ❌ Tool '{tool_name}' FAILED: {e}")
    
    entry["latency_ms"] = int((time.time() - start) * 1000)
    execution_log.append(entry)
    return entry

print("✅ Tool registry and execution logger ready")
print(f"Available tools: {list(TOOL_REGISTRY.keys())}")

## Step 3 — Simulate Agent Run (with deliberate errors for evaluation)

We simulate a realistic agent run with:
- Correct tool calls ✅
- Wrong tool chosen ❌  
- Hallucinated tool name 🚨
- Duplicate/redundant calls ⚠️
- Failed call (bad arguments) ❌

In [ ]:
execution_log.clear()  # reset log for fresh run

print("=" * 60)
print("SIMULATED AGENT RUN — Task: 'Check John's drug interaction'")
print("=" * 60)

# ── Call 1: ✅ Correct tool, correct args ─────────────────────
print("\n[Call 1] Fetch John's medication records")
tracked_tool_call(
    tool_name     = "patient_db",
    args          = {"patient_name": "John", "record_type": "medications"},
    expected_tool = "patient_db"
)

# ── Call 2: ✅ Correct tool, correct args ─────────────────────
print("\n[Call 2] Check drug interaction")
tracked_tool_call(
    tool_name     = "drug_interaction",
    args          = {"drug1": "Metformin", "drug2": "Amlodipine"},
    expected_tool = "drug_interaction"
)

# ── Call 3: ⚠️ REDUNDANT — same as Call 1 ────────────────────
print("\n[Call 3] Fetch John's records AGAIN (redundant)")
tracked_tool_call(
    tool_name     = "patient_db",
    args          = {"patient_name": "John", "record_type": "medications"},
    expected_tool = "patient_db"
)

# ── Call 4: ❌ WRONG TOOL selected ───────────────────────────
print("\n[Call 4] Agent should use patient_db but used web_search instead")
tracked_tool_call(
    tool_name     = "web_search",            # wrong tool for patient data
    args          = {"query": "John medications"},
    expected_tool = "patient_db"             # what SHOULD have been called
)

# ── Call 5: 🚨 HALLUCINATED tool name ────────────────────────
print("\n[Call 5] Agent calls a non-existent tool")
tracked_tool_call(
    tool_name     = "SecretMedicalDatabase.get_all_records",  # doesn't exist!
    args          = {"patient_id": "john_001"},
    expected_tool = "patient_db"
)

# ── Call 6: ❌ CORRECT tool, WRONG args → exception ──────────
print("\n[Call 6] Correct tool but hallucinated patient name")
tracked_tool_call(
    tool_name     = "patient_db",
    args          = {"patient_name": "pt_9999", "record_type": "medications"}, # ID doesn't exist
    expected_tool = "patient_db"
)

# ── Call 7: ✅ Correct tool, correct args ─────────────────────
print("\n[Call 7] Calculate CVD risk")
tracked_tool_call(
    tool_name     = "risk_calculator",
    args          = {"age": 45, "hba1c": 8.1, "bp_systolic": 150},
    expected_tool = "risk_calculator"
)

print(f"\nTotal tool calls made: {len(execution_log)}")

## Metric 1 — Tool Selection Accuracy

> Of all tool calls, what fraction chose the **correct tool** for the task?

$$\text{Tool Selection Accuracy} = \frac{\text{correct tool calls}}{\text{total tool calls with expected label}}$$

In [ ]:
labeled = [e for e in execution_log if e["expected_tool"] is not None]
correct = [e for e in labeled if e["tool_name"] == e["expected_tool"]]

tool_selection_accuracy = len(correct) / max(len(labeled), 1)

print("METRIC 1 — TOOL SELECTION ACCURACY")
print("=" * 50)
for e in labeled:
    match = e["tool_name"] == e["expected_tool"]
    status = "✅" if match else "❌"
    print(f"  {status} Called: '{e['tool_name']:35s}' | Expected: '{e['expected_tool']}'")

print(f"\nTool Selection Accuracy = {len(correct)}/{len(labeled)} = {tool_selection_accuracy:.2f}")
print(f"{'✅ GOOD' if tool_selection_accuracy >= 0.9 else '⚠️  NEEDS IMPROVEMENT' if tool_selection_accuracy >= 0.7 else '🚨 POOR'}")

## Metric 2 — Tool Hallucination Rate

> What fraction of tool calls used **invented tool names** that don't exist in the registry?

$$\text{Hallucination Rate} = \frac{\text{calls to non-existent tools}}{\text{total calls}}$$

In [ ]:
total       = len(execution_log)
hallucinated = [e for e in execution_log if e["is_hallucinated"]]
hallucination_rate = len(hallucinated) / max(total, 1)

print("METRIC 2 — TOOL HALLUCINATION RATE")
print("=" * 50)
for e in execution_log:
    status = "🚨 HALLUCINATED" if e["is_hallucinated"] else "✅ real tool"
    print(f"  {status}: '{e['tool_name']}'")

print(f"\nHallucination Rate = {len(hallucinated)}/{total} = {hallucination_rate:.2f}")
print(f"{'✅ GOOD' if hallucination_rate <= 0.05 else '🚨 HIGH — use Function Calling to fix'}")

## Metric 3 — Tool Success Rate & Metric 4 — Tool Call Efficiency

**Success Rate**: What % of real tool calls completed without exception?  
**Efficiency**: Did the agent make unnecessary / duplicate calls?

In [ ]:
# ── Metric 3: Tool Success Rate ──────────────────────────────────────────────
real_calls    = [e for e in execution_log if not e["is_hallucinated"]]
succeeded     = [e for e in real_calls if e["success"]]
success_rate  = len(succeeded) / max(len(real_calls), 1)

print("METRIC 3 — TOOL SUCCESS RATE")
print("=" * 50)
for e in real_calls:
    status = "✅ success" if e["success"] else f"❌ FAILED: {e['error'][:50]}"
    print(f"  [{e['call_id']}] {e['tool_name']:20s} → {status}")

print(f"\nTool Success Rate = {len(succeeded)}/{len(real_calls)} = {success_rate:.2f}")
print(f"{'✅ GOOD' if success_rate >= 0.95 else '⚠️  CHECK ARGUMENT VALIDATION'}")

# ── Metric 4: Tool Call Efficiency ───────────────────────────────────────────
print("\n\nMETRIC 4 — TOOL CALL EFFICIENCY")
print("=" * 50)

# Detect duplicate calls (same tool + same args)
seen_calls = {}
duplicates = 0
for e in execution_log:
    key = (e["tool_name"], str(e["args"]))
    if key in seen_calls:
        duplicates += 1
        print(f"  ⚠️  DUPLICATE: '{e['tool_name']}' with args {e['args']} (already called as [{seen_calls[key]}])")
    else:
        seen_calls[key] = e["call_id"]

unique_calls   = total - duplicates
optimal_calls  = 4    # minimum needed for this task: patient_db + drug_interaction + risk_calculator + web_search
efficiency     = optimal_calls / max(total, 1)
redundancy_rate = duplicates / max(total, 1)

print(f"\nTotal calls     : {total}")
print(f"Duplicate calls : {duplicates}")
print(f"Optimal calls   : {optimal_calls} (minimum needed)")
print(f"Efficiency      = {optimal_calls}/{total} = {efficiency:.2f}")
print(f"Redundancy Rate = {duplicates}/{total} = {redundancy_rate:.2f}")
print(f"{'✅ EFFICIENT' if efficiency >= 0.8 else '⚠️  TOO MANY REDUNDANT CALLS'}")

## Metric 5 — Tool Output Utilization

> Did the agent actually **use** the tool's output in its final answer?  
> A tool called but ignored = wasted API call + tokens.

In [ ]:
# Simulate: agent's final answer
final_answer = """
John takes Metformin 500mg twice daily and Amlodipine 5mg.
No major drug interaction found — potassium monitoring recommended.
His 10-year CVD risk is estimated at HIGH level based on age, HbA1c and BP.
"""

def check_output_utilized(tool_entry: dict, answer: str) -> bool:
    """Check if key data from tool result appears in the final answer."""
    if not tool_entry["success"] or tool_entry["result"] is None:
        return False
    result_str = str(tool_entry["result"]).lower()
    # Extract key words from tool result
    key_words = [w for w in result_str.split() if len(w) > 4][:5]
    answer_lower = answer.lower()
    matches = sum(1 for w in key_words if w in answer_lower)
    return matches >= 2  # at least 2 key words appear in answer

print("METRIC 5 — TOOL OUTPUT UTILIZATION")
print("=" * 50)
utilized = 0
successful_calls = [e for e in execution_log if e["success"]]

for e in successful_calls:
    used = check_output_utilized(e, final_answer)
    e["output_used_in_answer"] = used
    utilized += int(used)
    status = "✅ USED" if used else "⚠️  IGNORED"
    print(f"  {status}: '{e['tool_name']}' output")

utilization_rate = utilized / max(len(successful_calls), 1)
print(f"\nOutput Utilization = {utilized}/{len(successful_calls)} = {utilization_rate:.2f}")
print(f"{'✅ GOOD' if utilization_rate >= 0.85 else '⚠️  Some tool outputs were ignored'}")

## Full Evaluation Report — All Metrics Together

In [ ]:
import pandas as pd

# ── Compute all metrics ───────────────────────────────────────────────────────
total_calls       = len(execution_log)
real_calls        = [e for e in execution_log if not e["is_hallucinated"]]
succeeded_calls   = [e for e in real_calls if e["success"]]
labeled_calls     = [e for e in execution_log if e["expected_tool"] is not None]
correct_calls     = [e for e in labeled_calls if e["tool_name"] == e["expected_tool"]]
hallucinated_calls= [e for e in execution_log if e["is_hallucinated"]]

# Duplicate detection
call_signatures   = [(e["tool_name"], str(e["args"])) for e in execution_log]
duplicate_count   = total_calls - len(set(call_signatures))
optimal_call_count= 4  # minimum needed

metrics = {
    "Metric"         : [
        "Tool Selection Accuracy",
        "Tool Hallucination Rate",
        "Tool Success Rate",
        "Tool Call Efficiency",
        "Redundancy Rate",
        "Output Utilization",
    ],
    "Score" : [
        round(len(correct_calls) / max(len(labeled_calls), 1), 2),
        round(len(hallucinated_calls) / max(total_calls, 1), 2),
        round(len(succeeded_calls) / max(len(real_calls), 1), 2),
        round(optimal_call_count / max(total_calls, 1), 2),
        round(duplicate_count / max(total_calls, 1), 2),
        round(utilization_rate, 2),
    ],
    "Target": ["> 0.90", "< 0.05", "> 0.95", "> 0.80", "< 0.10", "> 0.85"],
    "Status": []
}

thresholds = [(0.90, True), (0.05, False), (0.95, True), (0.80, True), (0.10, False), (0.85, True)]
for score, higher_is_better in zip(metrics["Score"], thresholds[::1]):
    threshold, hib = thresholds[metrics["Score"].index(score)]
    good = score >= threshold if hib else score <= threshold
    metrics["Status"].append("✅ GOOD" if good else "🚨 NEEDS FIX")

df = pd.DataFrame(metrics)

print("=" * 65)
print("COMPLETE TOOL EVALUATION REPORT")
print("=" * 65)
print(df.to_string(index=False))

print("\n\nPer-call breakdown:")
print("-" * 65)
for e in execution_log:
    status = "✅" if e["success"] else ("🚨" if e["is_hallucinated"] else "❌")
    correct = "✅ correct" if e["is_correct_tool"] else ("❌ wrong" if e["is_correct_tool"] is False else "")
    print(f"  {status} [{e['call_id']}] {e['tool_name']:38s} {correct}")

## LangSmith — Automatic Tool Tracing (Production Way)

With `LANGCHAIN_TRACING_V2=true` in your `.env`, LangSmith captures all tool calls automatically.

```python
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.tools import tool
from langchain import hub

load_dotenv()  # loads LANGCHAIN_API_KEY, LANGCHAIN_PROJECT etc from .env

# Define tools with @tool decorator — LangSmith traces every call automatically
@tool
def patient_db(patient_name: str, record_type: str = "all") -> str:
    """Query patient medical records by patient name and record type."""
    # ... your implementation
    return f"John: Metformin 500mg, Amlodipine 5mg"

@tool  
def drug_interaction(drug1: str, drug2: str) -> str:
    """Check interaction between two drugs."""
    return "No major interaction. Monitor potassium."

# Create agent with tools
llm   = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [patient_db, drug_interaction]

# With LangSmith tracing ON:
# Every tool call → automatically logged to LangSmith dashboard
# You can see: tool name, args, output, latency, errors — all per run
agent = create_openai_functions_agent(llm, tools, hub.pull("hwchase17/openai-functions-agent"))
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = executor.invoke({"input": "Check John's drug interactions"})
# → LangSmith dashboard shows full trace at https://smith.langchain.com
```

**What LangSmith gives you for free**:
- Every tool call logged (name, args, output, latency)
- Tool success/failure per run  
- Token usage per tool call
- Full agent trajectory visualization
- Compare runs side-by-side

## Summary — Complete Reference

| Metric | Formula | Target | Fix if Low |
|---|---|---|---|
| **Tool Selection Accuracy** | correct / labeled | > 0.90 | Better router, clearer tool descriptions |
| **Tool Hallucination Rate** | hallucinated / total | < 0.05 | Use Function Calling (OpenAI) |
| **Tool Success Rate** | succeeded / real_calls | > 0.95 | Validate args with Pydantic |
| **Tool Call Efficiency** | optimal / actual | > 0.80 | Detect+skip duplicate calls |
| **Redundancy Rate** | duplicates / total | < 0.10 | Cache tool results in STM |
| **Output Utilization** | used / succeeded | > 0.85 | Check if agent reads STM outputs |

## The 3 Most Important

```
1. Hallucination Rate   ← use Function Calling → structurally prevents fake tool names
2. Tool Selection Accuracy ← right tool = right data = right answer
3. Tool Success Rate    ← validate args with Pydantic before execution
```